# From ECG to HR and simple HRV (Principles: vagal tone, RSA)

## TODOs

1. Compute RMSSD and HF power during baseline vs stress masks. 
2. Describe which increases/decreases and why (vagal withdrawal during stress → lower RMSSD/HF).

## My notes

+ RMSSD (Root Mean Square of Successive Differences)
    - Calculate using the R-peaks, where these R-peaks are the heartbeats.
    - R-peaks come from ECG, where ECG has P-wave, QRS-complex, and T-wave.
    - Time domain.
    - Steps:
        1. From the ECG, detect the components: P-wave, QRS-complex, and T-wave. 
        2. Delineate ECG for R-peaks, where these R-peaks are the heartbeats
        3. Calculate RMSSD by passing in R-peaks to get HRV
            - 3.1 1 R-peak = 1 Heartbeat
            - 3.2 Consecutive R-peaks are RR-interval = consecutive HBs
                - 3.2.1  From RR-interval, calculate LF/HF
    - We want this value to be
        1. high bc it leads to better recovery + more ANS stimulaton?
            - high means more time between heartbeats
        2. varied bc it leads to better recovery
+ HF (High Frequency) power
    1. Frequency domain.
    2. Favors parasympathetic activity, which is rest and recovery/digest.
    3. We want this value to be high bc...

## Code

In [1]:
import os
import pickle

import numpy as np
import pandas as pd

import neurokit2 as nk

import matplotlib.pyplot as plt

from tqdm import tqdm

from scipy.interpolate import interp1d
from scipy.signal import find_peaks, welch
from adtk.data import validate_series
from adtk.detector import AutoregressionAD, ThresholdAD

notebook_dir = os.getcwd()

## Load Data

In [2]:
ecg_sampling_rate = 700

In [3]:
data_path = os.path.join(notebook_dir, "../../../data/WESAD/all_subjects/chest.csv")
df = pd.read_csv(data_path)
df.drop(['Unnamed: 0'], axis=1, inplace=True)
df

,ACC_x,ACC_y,ACC_z,ECG,EMG,EDA,Temp,Resp,subject,label
0,0.9554,-0.2220,-0.5580,0.021423,-0.004440,5.250549,30.120758,-1.148987,S2,0
1,0.9258,-0.2216,-0.5538,0.020325,0.004349,5.267334,30.129517,-1.124573,S2,0
2,0.9082,-0.2196,-0.5392,0.016525,0.005173,5.243301,30.138214,-1.152039,S2,0
3,0.8974,-0.2102,-0.5122,0.016708,0.007187,5.249405,30.129517,-1.158142,S2,0
4,0.8882,-0.2036,-0.4824,0.011673,-0.015152,5.286407,30.130950,-1.161194,S2,0
...,...,...,...,...,...,...,...,...,...,...
60807595,0.9006,-0.0400,-0.1898,0.173676,-0.005539,7.265091,33.862762,0.318909,S17,0
60807596,0.9022,-0.0398,-0.1872,0.168777,-0.004944,7.266617,33.859740,0.233459,S17,0
60807597,0.9028,-0.0424,-0.1864,0.167130,-0.016068,7.262802,33.864290,0.318909,S17,0
60807598,0.9002,-0.0408,-0.1854,0.170334,0.005173,7.269669,33.862762,0.308228,S17,0


## Get Subject of Interest + Health Metric of Interest

In [4]:
def get_per_subject(df):
    per_subject = {}
    entries = df['subject'].unique()
    for entry in entries:
        filt_subject = (df['subject'] == entry)
        subject_df = df[filt_subject]
        subject_df.reset_index(inplace=True)
        per_subject[entry] = subject_df
    return per_subject

In [5]:
per_subject_dfs = get_per_subject(df)
per_subject_dfs

{'S2':            index   ACC_x   ACC_y   ACC_z       ECG       EMG       EDA  \
 0              0  0.9554 -0.2220 -0.5580  0.021423 -0.004440  5.250549   
 1              1  0.9258 -0.2216 -0.5538  0.020325  0.004349  5.267334   
 2              2  0.9082 -0.2196 -0.5392  0.016525  0.005173  5.243301   
 3              3  0.8974 -0.2102 -0.5122  0.016708  0.007187  5.249405   
 4              4  0.8882 -0.2036 -0.4824  0.011673 -0.015152  5.286407   
 ...          ...     ...     ...     ...       ...       ...       ...   
 4255295  4255295  0.8750 -0.1234 -0.2974 -0.013138  0.020370  0.400162   
 4255296  4255296  0.8750 -0.1262 -0.2988 -0.010345  0.019592  0.355911   
 4255297  4255297  0.8718 -0.1238 -0.3042 -0.005447 -0.017166  0.360489   
 4255298  4255298  0.8730 -0.1234 -0.3026  0.000137 -0.028976  0.365829   
 4255299  4255299  0.8702 -0.1220 -0.3022  0.004074 -0.023575  0.365448   
 
               Temp      Resp subject  label  
 0        30.120758 -1.148987      S2      0 

In [6]:
s_df = per_subject_dfs['S7']
ecg_df = s_df.loc[:, ['ECG']]
ecg_df

,ECG
0,-0.019226
1,-0.016937
2,-0.016159
3,-0.012405
4,-0.020325
...,...
3666595,0.156418
3666596,0.115723
3666597,0.086380
3666598,0.139938


## Create Time Component

In [7]:
milliseconds_per_sample = (1000 / ecg_sampling_rate)
milliseconds_per_sample

1.4285714285714286

In [8]:

# # Compute elapsed time in milliseconds
# ecg_df['Milliseconds'] = ecg_df.index * milliseconds_per_sample

# # Optionally, create a datetime-like column starting at 0
# ecg_df['Time'] = pd.to_timedelta(ecg_df['Milliseconds'], unit='ms')

# ecg_df


# milliseconds_per_sample = 1000 / ecg_sampling_rate
# ecg_df['Milliseconds'] = (ecg_df.index * milliseconds_per_sample).round(2)
# ecg_df['Seconds'] = pd.to_timedelta(ecg_df['Milliseconds'], unit='ms')
# ecg_df


# Elapsed time from start
ecg_df['Milliseconds'] = (ecg_df.index * milliseconds_per_sample).round(2)

# ✅ Numeric seconds for masks & segmentation
ecg_df['Seconds'] = (ecg_df['Milliseconds'] / 1000.0).round(2)

# Optional: display-only Timedelta (rounded to 10 ms so it matches two-decimal seconds)
ecg_df['Time'] = pd.to_timedelta(ecg_df['Seconds'], unit='s').dt.round('10ms')
ecg_df


,ECG,Milliseconds,Seconds,Time
0,-0.019226,0.00,0.00,0 days 00:00:00
1,-0.016937,1.43,0.00,0 days 00:00:00
2,-0.016159,2.86,0.00,0 days 00:00:00
3,-0.012405,4.29,0.00,0 days 00:00:00
4,-0.020325,5.71,0.01,0 days 00:00:00.010000
...,...,...,...,...
3666595,0.156418,5237992.86,5237.99,0 days 01:27:17.990000
3666596,0.115723,5237994.29,5237.99,0 days 01:27:17.990000
3666597,0.086380,5237995.71,5238.00,0 days 01:27:18
3666598,0.139938,5237997.14,5238.00,0 days 01:27:18


### Subset of Data (Optional)

In [9]:
def get_first_n_subset(df, subset_data_N):
    subset = df.loc[:subset_data_N, :]
    # print(len(subset), subset)

    return subset

# subset_df = get_first_n_subset(ecg_df, 10000)
# subset_df

## Clean Data

In [10]:
def get_cleaned_data(df, col_name, hz: int = 700):
    cleaned_series = nk.ecg_clean(df[col_name], hz)
    df[f'Cleaned {col_name}'] = cleaned_series

    return df

# get_cleaned_data(subset_df, 'ECG')
get_cleaned_data(ecg_df, 'ECG', ecg_sampling_rate)

,ECG,Milliseconds,Seconds,Time,Cleaned ECG
0,-0.019226,0.00,0.00,0 days 00:00:00,0.042484
1,-0.016937,1.43,0.00,0 days 00:00:00,0.035891
2,-0.016159,2.86,0.00,0 days 00:00:00,0.029329
3,-0.012405,4.29,0.00,0 days 00:00:00,0.022990
4,-0.020325,5.71,0.01,0 days 00:00:00.010000,0.017082
...,...,...,...,...,...
3666595,0.156418,5237992.86,5237.99,0 days 01:27:17.990000,-0.278476
3666596,0.115723,5237994.29,5237.99,0 days 01:27:17.990000,-0.261208
3666597,0.086380,5237995.71,5238.00,0 days 01:27:18,-0.242729
3666598,0.139938,5237997.14,5238.00,0 days 01:27:18,-0.222802


## Split Data into Segments

**Purpose:** We want to compare a single patient health metric over time such that we can track outliers. Splitting into segments allows us to have windows of the health metric. We track changes from each window depending on specific health metric.

In [11]:
def get_segments_by_counts(df, col_name, n_segments) -> pd.DataFrame:
    series = df.loc[:, col_name].values
    segments = np.array_split(series, n_segments)
    segments_df = pd.DataFrame(segments)
    return segments_df

# segments_df = get_segments(subset_df, 'Cleaned ECG', 3)
# segments_df = get_segments_by_counts(ecg_df, 'Cleaned ECG', 174)
# segments_df

In [12]:

def get_segments_by_duration(df, window_size_sec: int):
    # Work on a copy to avoid chained assignment issues
    out = df.copy()

    # window_id from numeric seconds
    out['window_id'] = (out['Seconds'] // window_size_sec).astype(int)

    # window start/end in seconds (numeric)
    out['window_start_seconds'] = out['window_id'] * window_size_sec
    out['window_end_seconds']   = out['window_start_seconds'] + window_size_sec

    # Per-window sample counts (sanity check)
    samples_per_window = out.groupby('window_id').size()

    return out, samples_per_window

segments_df, segments_groupby_df = get_segments_by_duration(ecg_df, 60)


In [13]:
segments_df

,ECG,Milliseconds,Seconds,Time,Cleaned ECG,window_id,window_start_seconds,window_end_seconds
0,-0.019226,0.00,0.00,0 days 00:00:00,0.042484,0,0,60
1,-0.016937,1.43,0.00,0 days 00:00:00,0.035891,0,0,60
2,-0.016159,2.86,0.00,0 days 00:00:00,0.029329,0,0,60
3,-0.012405,4.29,0.00,0 days 00:00:00,0.022990,0,0,60
4,-0.020325,5.71,0.01,0 days 00:00:00.010000,0.017082,0,0,60
...,...,...,...,...,...,...,...,...
3666595,0.156418,5237992.86,5237.99,0 days 01:27:17.990000,-0.278476,87,5220,5280
3666596,0.115723,5237994.29,5237.99,0 days 01:27:17.990000,-0.261208,87,5220,5280
3666597,0.086380,5237995.71,5238.00,0 days 01:27:18,-0.242729,87,5220,5280
3666598,0.139938,5237997.14,5238.00,0 days 01:27:18,-0.222802,87,5220,5280


In [14]:
segments_groupby_df

window_id
0     41997
1     42000
2     42000
3     42000
4     42000
      ...  
83    42000
84    42000
85    42000
86    42000
87    12603
Length: 88, dtype: int64

In [15]:
windows_df = segments_df[['window_id', 'window_start_seconds', 'window_end_seconds']].drop_duplicates()
windows_df = windows_df.sort_values('window_id').reset_index(drop=True)
windows_df

,window_id,window_start_seconds,window_end_seconds
0,0,0,60
1,1,60,120
2,2,120,180
3,3,180,240
4,4,240,300
...,...,...,...
83,83,4980,5040
84,84,5040,5100
85,85,5100,5160
86,86,5160,5220


## Find the R-peaks

- Also known as heartbeats
- R-peaks come from delineating the ECG signal into P-wave, QRS-complex, T-wave

In [40]:

# Choose the signal column
signal_col = 'Cleaned ECG' if 'Cleaned ECG' in segments_df.columns else 'ECG'

# For fast slicing, use a sorted numeric seconds array once
seconds = segments_df['Seconds'].to_numpy()
signal  = segments_df[signal_col].to_numpy()
print(len(signal))

r_peaks = []

for _, segment in tqdm(windows_df.iterrows(), total=len(windows_df)):
    start_time = float(segment['window_start_seconds'])
    end_time   = float(segment['window_end_seconds'])
    segment_idx =  segment['window_id']
    # print(f"Segment ({segment_idx})\n start time: {start_time} | end time: {end_time}")
    
    # Use the time window to get the index
    start_index = seconds.searchsorted(start_time, side='left')
    end_index   = seconds.searchsorted(end_time,   side='left')
    # print(f" start idx: {start_index} | end idx: {end_index}")

    # Use the indicies to get the signal's segment
    segment_ecg = signal[start_index:end_index]
    # print(f" segment idx: ({start_index} : {end_index}) | segment length: {len(segment_ecg)}\n\tsegment: {segment_ecg}")

    if segment_idx < 3:
        print(f"Segment ({segment_idx})\n start time: {start_time} | end time: {end_time}")
        print(f" start idx: {start_index} | end idx: {end_index}")
        print(f" segment idx: ({start_index} : {end_index}) | segment length: {len(segment_ecg)}\n\tsegment values: {segment_ecg}")

    # If segment is empty
    if segment_ecg.size == 0:
        r_peaks.append({
            'window_id': int(segment['window_id']),
            'window_start_seconds': start_time,
            'window_end_seconds': end_time,
            'n_samples': 0,
            'n_r_peaks': 0,
            'r_peaks_samples': np.array([], dtype=int),
            'note': 'empty window',
        })
        continue

    # If segment is NOT empty
    info = nk.ecg_findpeaks(segment_ecg, sampling_rate=ecg_sampling_rate)
    peaks = info.get('ECG_R_Peaks', np.array([], dtype=int))

    if segment_idx < 3:
        print(f"\t#peaks: {len(peaks)}")

    r_peaks.append({
        'window_id': int(segment['window_id']),
        'window_start_seconds': start_time,
        'window_end_seconds': end_time,
        'n_samples': int(segment_ecg.size),
        'n_r_peaks': int(peaks.size),
        'r_peaks_samples': peaks,  # indices are relative to the segment
    })


3666600


100%|██████████| 88/88 [00:00<00:00, 1159.23it/s]

Segment (0)
 start time: 0.0 | end time: 60.0
 start idx: 0 | end idx: 41997
 segment idx: (0 : 41997) | segment length: 41997
	segment values: [ 0.04248399  0.03589075  0.02932948 ... -0.04887366 -0.04819922
 -0.04749348]
	#peaks: 77
Segment (1)
 start time: 60.0 | end time: 120.0
 start idx: 41997 | end idx: 83997
 segment idx: (41997 : 83997) | segment length: 42000
	segment values: [-0.04674359 -0.04597735 -0.04525315 ... -0.07513546 -0.07332525
 -0.0717312 ]
	#peaks: 70
Segment (2)
 start time: 120.0 | end time: 180.0
 start idx: 83997 | end idx: 125997
 segment idx: (83997 : 125997) | segment length: 42000
	segment values: [-0.07031127 -0.06910377 -0.06824274 ...  0.15489931  0.2183424
  0.29020012]
	#peaks: 70


In [29]:
r_peaks

[]

## Get Heart Rate Variability (HRV) Metrics

- Use the R-peaks

In [ ]:
import warnings
warnings.filterwarnings('ignore')
hrv_stats = []
for r_peaks_idx in range(len(r_peaks)):
    r_peaks_info = r_peaks[r_peaks_idx]
    r_preak_name = r_peaks_info['window_id']
    r_peaks_per_segment = r_peaks_info['r_peaks_samples']
    print(r_peaks_per_segment)

    hrv_segment_stats = nk.hrv_time(r_peaks_per_segment, ecg_sampling_rate, show=False)
    collect_hrv_stats = {
        f'HRV Stats @ {r_preak_name}': hrv_segment_stats
    }
    if r_peaks_idx < 2:
        print(f"{r_preak_name}: {r_peaks_per_segment}")
        print(f"\tstats: {hrv_segment_stats.to_dict()}\n")
    hrv_stats.append(hrv_segment_stats)

In [ ]:
segment_stats_df = pd.concat(hrv_stats)
segment_stats_df.reset_index(inplace=True)
segment_stats_df

In [ ]:
segment_stats_df.columns.to_list()

### HRV: RMSSD

In [ ]:
rmssd_series = segment_stats_df.loc[:, 'HRV_RMSSD']
rmssd_series

In [ ]:
rmssd_series.index = pd.to_datetime(rmssd_series.index)
rmssd_series

In [ ]:
rmssd_series.plot.line()

In [ ]:
# Statistical approach
mean_amp = np.mean(rmssd_series)
std_amp = np.std(rmssd_series)
low_threshold = mean_amp - 3*std_amp
high_threshold = mean_amp + 3*std_amp


anomoly_detector = ThresholdAD(low=low_threshold, high=high_threshold)
anomolies = anomoly_detector.detect(rmssd_series, True)
anomolies

## Follow-Up

> - Be sure to see TODOs

1. What do we gather from this subject?
2. Is there anything to flag?
3. 'RMSSD_ms': np.float64(698.4470516017469)
    - Why?
    - Good/Bad?
4. 'HF': np.float64(0.04114562090390898), why?
    - Why?
    - Good/Bad?
5. RMSSD_ms and HF considered jointly
    - If one is good and other bad? Vice versa
    - If both are good? Both are bad?

### 1. What do we gather from this subject?

> - R-peaks (aka heartbeats in green).


In [ ]:
# Could say something based on the avg of the window of 60 secs or take last

# Choose sampling rate and adjustable
# 